In [2]:
import os, json, datetime
from arxiv_provider import search_arxiv
from enhanced_query_script import to_jsonl, to_csv

queries_path = "../queries/queries_arxiv.json"  # the JSON array you built earlier
with open(queries_path, "r", encoding="utf-8") as f:
    queries = json.load(f)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
outdir = os.path.join("../outputs", f"arxiv_run_{ts}")
os.makedirs(outdir, exist_ok=True)

for idx, q in enumerate(queries, start=1):
    print(f"\n=== arXiv Query {idx}/{len(queries)} ===")
    print(q)
    results = search_arxiv(
        q,
        year_min=2018,
        categories=["cs.CV", "cs.LG", "stat.ML", "eess.IV"],  # adjust or set None to skip
        per_page=100,
        sort_by="submittedDate",
        sort_order="descending",
        polite_delay=3.0,  # keep this >= 3s for arXiv politeness
    )
    for r in results:
        r["query"] = q
        r["query_id"] = f"Q{idx:02d}"
    to_jsonl(os.path.join(outdir, f"Q{idx:02d}_results.jsonl"), results)
    to_csv  (os.path.join(outdir, f"Q{idx:02d}_results.csv"),   results)
    print(f"Found {len(results)} results")


=== arXiv Query 1/16 ===
((ti:"field conditions" OR abs:"field conditions") AND (ti:"plant disease" OR abs:"plant disease" OR ti:pest OR abs:pest) AND (ti:image OR abs:image OR ti:imaging OR abs:imaging OR ti:vision OR abs:vision OR ti:camera OR abs:camera OR ti:rgb OR abs:rgb OR ti:hyperspectral OR abs:hyperspectral OR ti:UAV OR abs:UAV OR ti:drone OR abs:drone) AND (ti:adaptation OR abs:adaptation OR ti:adapt OR abs:adapt OR ti:generalization OR abs:generalization OR ti:generalisation OR abs:generalisation OR ti:robust OR abs:robust OR ti:robustness OR abs:robustness)) AND (cat:cs.CV OR cat:cs.LG OR cat:eess.IV)
Found 7 results

=== arXiv Query 2/16 ===
((ti:"in field" OR abs:"in field") AND (ti:plant OR abs:plant OR ti:crop OR abs:crop) AND (ti:disease OR abs:disease OR ti:pest OR abs:pest) AND (ti:image OR abs:image OR ti:imaging OR abs:imaging OR ti:vision OR abs:vision OR ti:camera OR abs:camera OR ti:rgb OR abs:rgb OR ti:hyperspectral OR abs:hyperspectral OR ti:UAV OR abs:UAV O

In [3]:
# aggregate all results
import glob
all_results = []
for fn in glob.glob(os.path.join(outdir, "Q??_results.jsonl")):
    with open(fn, "r", encoding="utf-8") as f:
        for line in f:
            all_results.append(json.loads(line))
print(f"Total aggregated results: {len(all_results)}")
to_jsonl(os.path.join(outdir, f"all_results.jsonl"), all_results)
to_csv  (os.path.join(outdir, f"all_results.csv"),   all_results)

Total aggregated results: 117
